In [1]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Context Graph V5: TTL Import, Mixed Extraction, Temporal Lineage

**End-to-end demo of the V5 pipeline: OWL/TTL import, structured + AI extraction,
batch materialization with status reporting, and cross-session lineage queries.**

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ontology_graph_v5_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/ontology_graph_v5_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32"> Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/ontology_graph_v5_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35"> Open in BQ Studio
    </a>
  </td>
</table>

## Overview

The **Context Graph V5** pipeline extends V4 with three new capabilities:

1. **TTL/OWL Import** -- bootstrap an ontology spec from an existing OWL/Turtle file,
   then resolve placeholder keys interactively.
2. **Mixed Extraction** -- combine deterministic structured extractors (e.g., BKA decision
   events) with AI-based extraction for the remaining telemetry.
3. **Temporal Lineage** -- detect property changes on shared-key entities across sessions
   and materialize cross-session lineage edges queryable via GQL.

Additional V5 improvements:

- **`batch_load` write mode** -- uses `load_table_from_json` with staging tables for
  more reliable materialization with full status reporting.
- **`materialize_with_status()`** -- returns `MaterializationResult` with per-table
  `TableStatus` (cleanup status, insert status, idempotency flag).
- **Scratch datasets** with auto-expiring tables for safe, isolated demos.

## Install Dependencies

In [2]:
!pip install -q bigquery-agent-analytics google-cloud-bigquery pyyaml rdflib

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621


ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement bigquery-agent-analytics (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip
ERROR: No matching distribution found for bigquery-agent-analytics


## Authenticate & Configure

In [3]:
import os

try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab -- using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")

Not running in Colab -- using default credentials.
Project  : test-project-0728-467323
Dataset  : agent_analytics
Table    : agent_events


In [4]:
import uuid
from google.cloud import bigquery

RUN_ID = uuid.uuid4().hex[:8]
SCRATCH_DATASET = f"{DATASET_ID}_v5_{RUN_ID}"
SCRATCH_ENV = f"{PROJECT_ID}.{SCRATCH_DATASET}"

bq = bigquery.Client(project=PROJECT_ID)
ds = bigquery.Dataset(f"{PROJECT_ID}.{SCRATCH_DATASET}")
ds.default_table_expiration_ms = 3600000  # 1h TTL
bq.create_dataset(ds, exists_ok=True)
print(f"Scratch dataset: {SCRATCH_DATASET}")
print(f"Tables auto-expire in 1 hour")

Scratch dataset: agent_analytics_v5_8ec7050a
Tables auto-expire in 1 hour


## Step 1: TTL Import

Import an OWL/Turtle ontology file into a GraphSpec-compatible YAML artifact.
The `ttl_import` function reads the `.ttl` file, filters by namespace, and generates
a YAML spec with entities, relationships, properties, and primary keys derived from
`owl:hasKey` declarations. Classes without `owl:hasKey` get a `FILL_IN` placeholder.

In [5]:
TTL_PATH = "yamo_sample.ttl"

_TTL_CONTENT = """@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix yamo: <https://example.com/yamo#> .

# ============================================================
# Classes
# ============================================================

yamo:Party a owl:Class ;
  rdfs:label "Party" ;
  owl:hasKey ( yamo:party_id ) .

yamo:AdUnit a owl:Class ;
  rdfs:label "Ad Unit" ;
  rdfs:subClassOf yamo:Party ;
  owl:hasKey ( yamo:ad_unit_id ) .

yamo:Campaign a owl:Class ;
  rdfs:label "Campaign" ;
  owl:hasKey ( yamo:campaign_id ) .

yamo:RejectionReason a owl:Class ;
  rdfs:label "Rejection Reason" ;
  owl:hasKey ( yamo:rejection_type ) .

# DecisionPoint has NO owl:hasKey -> should produce FILL_IN.
yamo:DecisionPoint a owl:Class ;
  rdfs:label "Decision Point" .

# ============================================================
# Datatype Properties
# ============================================================

yamo:party_id a owl:DatatypeProperty ;
  rdfs:domain yamo:Party ;
  rdfs:range xsd:string .

yamo:name a owl:DatatypeProperty ;
  rdfs:domain yamo:Party ;
  rdfs:range xsd:string .

yamo:ad_unit_id a owl:DatatypeProperty ;
  rdfs:domain yamo:AdUnit ;
  rdfs:range xsd:string .

yamo:campaign_id a owl:DatatypeProperty ;
  rdfs:domain yamo:Campaign ;
  rdfs:range xsd:string .

yamo:decision_id a owl:DatatypeProperty ;
  rdfs:domain yamo:DecisionPoint ;
  rdfs:range xsd:string .

yamo:decision_type a owl:DatatypeProperty ;
  rdfs:domain yamo:DecisionPoint ;
  rdfs:range xsd:string .

yamo:rejection_type a owl:DatatypeProperty ;
  rdfs:domain yamo:RejectionReason ;
  rdfs:range xsd:string .

# budget is xsd:decimal -> should narrow to double with a warning.
yamo:budget a owl:DatatypeProperty ;
  rdfs:domain yamo:Campaign ;
  rdfs:range xsd:decimal .

yamo:start_date a owl:DatatypeProperty ;
  rdfs:domain yamo:Campaign ;
  rdfs:range xsd:date .

# ============================================================
# Object Properties (relationships)
# ============================================================

yamo:evaluates a owl:ObjectProperty ;
  rdfs:label "evaluates" ;
  rdfs:domain yamo:DecisionPoint ;
  rdfs:range yamo:AdUnit .

yamo:rejectedBy a owl:ObjectProperty ;
  rdfs:label "rejected by" ;
  rdfs:domain yamo:AdUnit ;
  rdfs:range yamo:RejectionReason .
"""

with open(TTL_PATH, "w") as f:
    f.write(_TTL_CONTENT)

print(f"Wrote {TTL_PATH} ({len(_TTL_CONTENT)} bytes)")

Wrote yamo_sample.ttl (2454 bytes)


In [6]:
from bigquery_agent_analytics.ttl_importer import ttl_import

result = ttl_import(
    ttl_path=TTL_PATH,
    include_namespaces=["https://example.com/yamo#"],
)
print("=== Import Report ===")
print(f"Classes mapped: {result.report.classes_mapped}")
print(f"Properties mapped: {result.report.properties_mapped}")
print(f"Relationships mapped: {result.report.relationships_mapped}")
print(f"Placeholders: {len(result.report.placeholders)}")
for p in result.report.placeholders:
    print(f"  - {p.location}: {p.reason}")
print(f"Type warnings: {len(result.report.type_warnings)}")
for w in result.report.type_warnings:
    print(f"  - {w.property_name}: {w.owl_type} -> {w.mapped_type}")


=== Import Report ===
Classes mapped: 5
Properties mapped: 9
Relationships mapped: 2
Placeholders: 1
  - entities[DecisionPoint].keys.primary: No owl:hasKey found; primary key must be specified.
Type warnings: 1
  - budget: http://www.w3.org/2001/XMLSchema#decimal -> double


/Users/haiyuancao/adk-python/src/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [7]:
# The full generated YAML -- note FILL_IN placeholders for classes without owl:hasKey
print(result.yaml_text)

ontology_import:
  status: unresolved
  source_file: /Users/haiyuancao/BigQuery-Agent-Analytics-SDK/examples/yamo_sample.ttl
  import_timestamp: '2026-04-14T05:58:24.248815+00:00'
  placeholders_remaining: 1
graph:
  name: yamo_imported
  entities:
  - name: AdUnit
    description: Ad Unit
    extends: Party
    binding:
      source: '{{ env }}.ad_units'
    keys:
      primary:
      - ad_unit_id
    properties:
    - name: ad_unit_id
      type: string
  - name: Campaign
    description: Campaign
    binding:
      source: '{{ env }}.campaigns'
    keys:
      primary:
      - campaign_id
    properties:
    - name: campaign_id
      type: string
    - name: budget
      type: double
    - name: start_date
      type: date
  - name: DecisionPoint
    description: Decision Point
    binding:
      source: '{{ env }}.decision_points'
    keys:
      primary:
      - FILL_IN
    properties:
    - name: decision_id
      type: string
    - name: decision_type
      type: string
  - name

### Resolve Placeholders

The `ttl_resolve` function takes the generated YAML text and a dict mapping
`FILL_IN` placeholders to actual primary key column names. This produces a
fully valid GraphSpec YAML.

In [8]:
import tempfile, os
from bigquery_agent_analytics.ttl_importer import ttl_resolve
from bigquery_agent_analytics.ontology_models import load_graph_spec_from_string

# Write unresolved artifact to a temp file for ttl_resolve.
with tempfile.NamedTemporaryFile(mode="w", suffix=".import.yaml", delete=False) as f:
    f.write(result.yaml_text)
    import_path = f.name

resolved = ttl_resolve(
    import_path,
    defaults={
        "entities[DecisionPoint].keys.primary": ["decision_id"],
        "relationships[evaluates].binding.from_columns": ["decision_id"],
    },
)
os.unlink(import_path)

spec_from_owl = load_graph_spec_from_string(resolved)
print(f"Resolved spec: {spec_from_owl.name}")
print(f"Entities: {[e.name for e in spec_from_owl.entities]}")
print(f"Relationships: {[r.name for r in spec_from_owl.relationships]}")


Resolved spec: yamo_imported
Entities: ['AdUnit', 'Campaign', 'DecisionPoint', 'Party', 'RejectionReason']
Relationships: ['evaluates', 'rejectedBy']


## Step 2: Mixed Extraction (Structured + AI)

V5 supports **mixed extraction**: deterministic structured extractors handle well-known
event types (e.g., BKA decision events), while AI.GENERATE covers the remaining
unstructured telemetry.

- Structured extractors declare which spans they fully or partially handle.
- The AI transcript excludes fully-handled spans, reducing cost and hallucination risk.
- Both paths produce `ExtractedNode` / `ExtractedEdge` instances that merge seamlessly.

In [9]:
from bigquery_agent_analytics.ontology_models import load_graph_spec

SPEC_PATH = "ymgo_graph_spec_v5.yaml"

_YMGO_SPEC_V5 = """\
graph:
  name: YMGO_Context_Graph_V3

  entities:
    - name: mako_DecisionPoint
      description: "The atomic unit of decisioning where an agent evaluates alternatives."
      binding:
        source: "{{ env }}.decision_points"
      keys:
        primary: [decision_id]
      properties:
        - name: decision_id
          type: string
        - name: decision_type
          type: string

    - name: sup_YahooAdUnit
      extends: mako_Candidate
      description: "A specific ad slot on a Yahoo property being evaluated as a candidate."
      binding:
        source: "{{ env }}.yahoo_ad_units"
      keys:
        primary: [adUnitId]
      properties:
        - name: adUnitId
          type: string
        - name: adUnitName
          type: string
        - name: adUnitSize
          type: string
        - name: adUnitPosition
          type: string

    - name: mako_RejectionReason
      description: "Structured reason why a Candidate was not selected at a DecisionPoint."
      binding:
        source: "{{ env }}.rejection_reasons"
      keys:
        primary: [rejection_id]
      properties:
        - name: rejection_id
          type: string
        - name: rejectionType
          type: string
        - name: rejectionRule
          type: string

  relationships:
    - name: CandidateEdge
      description: "Connects a decision point to the evaluated Yahoo Ad Unit."
      from_entity: mako_DecisionPoint
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.candidate_edges"
        from_columns: [decision_id]
        to_columns: [adUnitId]
      properties:
        - name: edge_type
          type: string
        - name: mako_scoreValue
          type: double

    - name: ForCandidate
      description: "Maps the MAKO rejection rationale directly to the dropped candidate."
      from_entity: mako_RejectionReason
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.rejection_mappings"
        from_columns: [rejection_id]
        to_columns: [adUnitId]

    - name: sup_YahooAdUnitEvolvedFrom
      description: "Tracks evolution of a YahooAdUnit across sessions."
      from_entity: sup_YahooAdUnit
      to_entity: sup_YahooAdUnit
      binding:
        source: "{{ env }}.sup_yahoo_ad_unit_lineage"
        from_columns: [adUnitId]
        to_columns: [adUnitId]
        from_session_column: from_session_id
        to_session_column: to_session_id
      properties:
        - name: from_session_id
          type: string
        - name: to_session_id
          type: string
        - name: event_time
          type: timestamp
        - name: changed_properties
          type: string
"""

with open(SPEC_PATH, "w") as f:
    f.write(_YMGO_SPEC_V5)
print(f"Wrote {SPEC_PATH}")

spec = load_graph_spec(SPEC_PATH, env=SCRATCH_ENV)
print(f"Graph: {spec.name}")
print(f"Entities: {[e.name for e in spec.entities]}")
print(f"Relationships: {[r.name for r in spec.relationships]}")


Wrote ymgo_graph_spec_v5.yaml
Graph: YMGO_Context_Graph_V3
Entities: ['mako_DecisionPoint', 'sup_YahooAdUnit', 'mako_RejectionReason']
Relationships: ['CandidateEdge', 'ForCandidate', 'sup_YahooAdUnitEvolvedFrom']


In [10]:
from bigquery_agent_analytics.structured_extraction import (
    extract_bka_decision_event,
    StructuredExtractor,
)

# Demo: run the BKA extractor on a synthetic event
demo_event = {
    "span_id": "span-001",
    "session_id": "demo-session",
    "event_type": "bka_decision",
    "content": {
        "decision_id": "dp_demo_eval",
        "outcome": "selected",
        "confidence": 0.92,
    },
}

demo_result = extract_bka_decision_event(demo_event, spec)
print(f"Nodes extracted: {len(demo_result.nodes)}")
for node in demo_result.nodes:
    print(f"  {node.node_id}")
    for p in node.properties:
        print(f"    {p.name} = {p.value}")
print(f"Fully handled spans: {demo_result.fully_handled_span_ids}")
print(f"Partially handled spans: {demo_result.partially_handled_span_ids}")

Nodes extracted: 1
  demo-session:mako_DecisionPoint:decision_id=dp_demo_eval
    decision_id = dp_demo_eval
    outcome = selected
    confidence = 0.92
Fully handled spans: {'span-001'}
Partially handled spans: set()


### Operational Note: AI Extraction Variance

The unstructured portion of this demo still relies on `AI.GENERATE`. That means extraction quality
can vary across models, prompts, and source traces.

The demo is stable because:
- deterministic structured extractors handle known event types first
- the notebook uses controlled fixtures and isolated scratch targets
- schema filtering drops extra AI-emitted fields before materialization

For real telemetry, treat AI-extracted graph content as model-dependent output rather than a
strictly deterministic parse.


In [11]:
from bigquery_agent_analytics.ontology_graph import OntologyGraphManager

SESSION_IDS = ["adcp-033c95d7a97d"]  # Replace with your session IDs

extractors = {"bka_decision": extract_bka_decision_event}

extractor = OntologyGraphManager(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    spec=spec,
    table_id=TABLE_ID,
    endpoint="gemini-2.5-flash",
    extractors=extractors,
)

graph = extractor.extract_graph(
    session_ids=SESSION_IDS,
    use_ai_generate=True,
)
print(f"Extracted {len(graph.nodes)} nodes, {len(graph.edges)} edges")
for node in graph.nodes[:5]:
    print(f"  [{node.entity_name}] {node.node_id}")
    for p in node.properties:
        print(f"    {p.name} = {p.value}")

Extracted 3 nodes, 2 edges
  [sup_YahooAdUnit] adcp-033c95d7a97d:sup_YahooAdUnit:adUnitId=yahoo_homepage_display_premium
    adUnitId = yahoo_homepage_display_premium
    adUnitName = Yahoo Homepage
    adUnitSize = display
    adUnitPosition = Premium Placement
  [sup_YahooAdUnit] adcp-033c95d7a97d:sup_YahooAdUnit:adUnitId=yahoo_mail_display_standard
    adUnitId = yahoo_mail_display_standard
    adUnitName = Yahoo Mail
    adUnitSize = display
    adUnitPosition = Standard
  [mako_DecisionPoint] adcp-033c95d7a97d:mako_DecisionPoint:decision_id=dp_allocate_media_budget_1
    decision_id = dp_allocate_media_budget_1
    decision_type = media_budget_allocation


## Step 3: Materialize with `batch_load`

The V5 materializer supports a `batch_load` write mode that uses `load_table_from_json`
with a staging table instead of the streaming `insert_rows_json` API. This provides:

- More reliable inserts for large batches.
- Full status reporting via `materialize_with_status()`.
- Per-table `TableStatus` with cleanup status, insert status, and idempotency flags.

In [12]:
from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer

materializer = OntologyMaterializer(
    project_id=PROJECT_ID,
    dataset_id=SCRATCH_DATASET,
    spec=spec,
    write_mode="batch_load",
)

# Create tables first
tables = materializer.create_tables()
print("Tables created:")
for name, ref in tables.items():
    print(f"  {name} -> {ref}")

# Materialize with full status reporting
result = materializer.materialize_with_status(graph, SESSION_IDS)
print(f"\nRows: {result.row_counts}")
for ref, ts in result.table_statuses.items():
    print(f"  {ref}: cleanup={ts.cleanup_status} insert={ts.insert_status} idempotent={ts.idempotent}")

Tables created:
  mako_DecisionPoint -> test-project-0728-467323.agent_analytics_v5_8ec7050a.decision_points
  sup_YahooAdUnit -> test-project-0728-467323.agent_analytics_v5_8ec7050a.yahoo_ad_units
  mako_RejectionReason -> test-project-0728-467323.agent_analytics_v5_8ec7050a.rejection_reasons
  CandidateEdge -> test-project-0728-467323.agent_analytics_v5_8ec7050a.candidate_edges
  ForCandidate -> test-project-0728-467323.agent_analytics_v5_8ec7050a.rejection_mappings
  sup_YahooAdUnitEvolvedFrom -> test-project-0728-467323.agent_analytics_v5_8ec7050a.sup_yahoo_ad_unit_lineage



Rows: {'sup_YahooAdUnit': 2, 'mako_DecisionPoint': 1, 'CandidateEdge': 2}
  test-project-0728-467323.agent_analytics_v5_8ec7050a.yahoo_ad_units: cleanup=deleted insert=inserted idempotent=True
  test-project-0728-467323.agent_analytics_v5_8ec7050a.decision_points: cleanup=deleted insert=inserted idempotent=True
  test-project-0728-467323.agent_analytics_v5_8ec7050a.candidate_edges: cleanup=deleted insert=inserted idempotent=True


In [13]:
from bigquery_agent_analytics.ontology_property_graph import compile_property_graph_ddl

ddl = compile_property_graph_ddl(spec, PROJECT_ID, SCRATCH_DATASET)
print(ddl)

CREATE OR REPLACE PROPERTY GRAPH `test-project-0728-467323.agent_analytics_v5_8ec7050a.YMGO_Context_Graph_V3`
  NODE TABLES (
    `test-project-0728-467323.agent_analytics_v5_8ec7050a.decision_points` AS mako_DecisionPoint
      KEY (decision_id, session_id)
      LABEL mako_DecisionPoint
      PROPERTIES (
        decision_id,
        decision_type,
        session_id,
        extracted_at
      ),
    `test-project-0728-467323.agent_analytics_v5_8ec7050a.yahoo_ad_units` AS sup_YahooAdUnit
      KEY (adUnitId, session_id)
      LABEL sup_YahooAdUnit
      LABEL mako_Candidate
      PROPERTIES (
        adUnitId,
        adUnitName,
        adUnitSize,
        adUnitPosition,
        session_id,
        extracted_at
      ),
    `test-project-0728-467323.agent_analytics_v5_8ec7050a.rejection_reasons` AS mako_RejectionReason
      KEY (rejection_id, session_id)
      LABEL mako_RejectionReason
      PROPERTIES (
        rejection_id,
        rejectionType,
        rejectionRule,
       

In [14]:
from bigquery_agent_analytics.ontology_property_graph import OntologyPropertyGraphCompiler

compiler = OntologyPropertyGraphCompiler(
    project_id=PROJECT_ID,
    dataset_id=SCRATCH_DATASET,
    spec=spec,
)
created = compiler.create_property_graph()
print(f"Property Graph created: {created}")

Property Graph created: True


## Step 4: Temporal Lineage

Detect property changes on shared-key entities across sessions. When the same
`adUnitId` appears in two sessions with different property values, lineage edges
are created to track the evolution.

We create a synthetic "Session B" with a modified `sup_YahooAdUnit` to demonstrate
cross-session lineage detection without requiring a second real extraction.

### Demo Note: Synthetic Lineage Input

The lineage walkthrough below uses a synthetic follow-up session that reuses a known `adUnitId` with modified properties.

This is intentional for demo determinism: it proves the end-to-end V5 lineage path
(`detect_lineage_edges` → materialization → `compile_lineage_gql` → BigQuery query execution)
without depending on real telemetry sessions having overlapping entity keys.

In production data, lineage edges will only appear when multiple sessions extract the same entity
primary key and at least one tracked property changes.


In [15]:
from bigquery_agent_analytics.ontology_models import (
    ExtractedGraph,
    ExtractedNode,
    ExtractedProperty,
)

session_a_ad_units = [n for n in graph.nodes if n.entity_name == "sup_YahooAdUnit"]
if session_a_ad_units:
    original = session_a_ad_units[0]
    original_props = {p.name: p.value for p in original.properties}
    shared_id = original_props.get("adUnitId", "unknown")

    synthetic_node = ExtractedNode(
        node_id=f"sess-synthetic-B:sup_YahooAdUnit:adUnitId={shared_id}",
        entity_name="sup_YahooAdUnit",
        labels=["sup_YahooAdUnit", "mako_Candidate"],
        properties=[
            ExtractedProperty(name="adUnitId", value=shared_id),
            ExtractedProperty(name="adUnitName", value=original_props.get("adUnitName", "") + " (Redesigned)"),
            ExtractedProperty(name="adUnitSize", value=original_props.get("adUnitSize", "300x250")),
            ExtractedProperty(name="adUnitPosition", value="BTF"),
        ],
    )
    synthetic_graph = ExtractedGraph(name=spec.name, nodes=[synthetic_node], edges=[])
    result_b = materializer.materialize_with_status(synthetic_graph, ["sess-synthetic-B"])
    print(f"Synthetic Session B: {result_b.row_counts}")
else:
    print("No ad units found in Session A -- skipping synthetic lineage demo")
    synthetic_graph = ExtractedGraph(name=spec.name, nodes=[], edges=[])

Synthetic Session B: {'sup_YahooAdUnit': 1}


In [16]:
from bigquery_agent_analytics.ontology_graph import detect_lineage_edges

lineage_edges = detect_lineage_edges(
    current_graph=synthetic_graph,
    current_session_id="sess-synthetic-B",
    prior_graphs={SESSION_IDS[0]: graph},
    lineage_entity_types=["sup_YahooAdUnit"],
    spec=spec,
)
print(f"Lineage edges detected: {len(lineage_edges)}")
for edge in lineage_edges:
    print(f"  {edge.edge_id}")
    for p in edge.properties:
        print(f"    {p.name} = {p.value}")

Lineage edges detected: 1
  sess-synthetic-B:sup_YahooAdUnitEvolvedFrom:adcp-033c95d7a97d:adUnitId=yahoo_homepage_display_premium
    from_session_id = adcp-033c95d7a97d
    to_session_id = sess-synthetic-B
    event_time = 2026-04-14T05:59:10.452196+00:00
    changed_properties = adUnitName,adUnitPosition


In [17]:
# Materialize lineage edges into the lineage table
lineage_graph = ExtractedGraph(
    name=spec.name,
    nodes=[],
    edges=lineage_edges,
)
result_lineage = materializer.materialize_with_status(lineage_graph, ["sess-synthetic-B"])
print(f"Lineage rows: {result_lineage.row_counts}")
for ref, ts in result_lineage.table_statuses.items():
    print(f"  {ref}: cleanup={ts.cleanup_status} insert={ts.insert_status} idempotent={ts.idempotent}")

Lineage rows: {'sup_YahooAdUnitEvolvedFrom': 1}
  test-project-0728-467323.agent_analytics_v5_8ec7050a.sup_yahoo_ad_unit_lineage: cleanup=deleted insert=inserted idempotent=True


## Step 5: GQL Queries

Query the materialized property graph using BigQuery GQL. We compile two queries:

1. **Showcase GQL** -- standard forward traversal from `DecisionPoint` to `YahooAdUnit`.
2. **Lineage GQL** -- cross-session lineage traversal via `sup_YahooAdUnitEvolvedFrom`.

In [18]:
from bigquery_agent_analytics.ontology_orchestrator import (
    compile_showcase_gql,
    compile_lineage_gql,
)

# Standard forward traversal GQL
showcase_gql = compile_showcase_gql(spec=spec, project_id=PROJECT_ID, dataset_id=SCRATCH_DATASET)
print("=== Showcase GQL ===")
print(showcase_gql)

# Lineage traversal GQL
lineage_gql = compile_lineage_gql(
    spec=spec,
    project_id=PROJECT_ID,
    dataset_id=SCRATCH_DATASET,
    relationship_name="sup_YahooAdUnitEvolvedFrom",
)
print("\n=== Lineage GQL ===")
print(lineage_gql)

=== Showcase GQL ===
GRAPH `test-project-0728-467323.agent_analytics_v5_8ec7050a.YMGO_Context_Graph_V3`
MATCH
  (dp:mako_DecisionPoint)-[ece:CandidateEdge]->(yau:sup_YahooAdUnit)
WHERE dp.session_id = @session_id
RETURN
  dp.decision_id AS src_decision_id,
  dp.decision_type AS src_decision_type,
  ece.edge_type,
  ece.mako_scoreValue,
  yau.adUnitId AS dst_adUnitId,
  yau.adUnitName AS dst_adUnitName,
  yau.adUnitSize AS dst_adUnitSize,
  yau.adUnitPosition AS dst_adUnitPosition
ORDER BY dp.decision_id
LIMIT @result_limit


=== Lineage GQL ===
GRAPH `test-project-0728-467323.agent_analytics_v5_8ec7050a.YMGO_Context_Graph_V3`
MATCH
  (prev:sup_YahooAdUnit)-[eyauef:sup_YahooAdUnitEvolvedFrom]->(cur:sup_YahooAdUnit)
WHERE cur.session_id = @session_id
RETURN
  prev.adUnitId AS prior_adUnitId,
  prev.adUnitName AS prior_adUnitName,
  prev.adUnitSize AS prior_adUnitSize,
  prev.adUnitPosition AS prior_adUnitPosition,
  eyauef.from_session_id,
  eyauef.to_session_id,
  eyauef.event_time,
  e

In [19]:
# Execute the forward traversal GQL
job = bq.query(
    showcase_gql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("session_id", "STRING", SESSION_IDS[0]),
            bigquery.ScalarQueryParameter("result_limit", "INT64", 50),
        ]
    ),
)
results = job.to_dataframe()
print(f"Forward traversal: {len(results)} rows")
if len(results) > 0:
    print(results.head(10))
else:
    print("No rows returned")

Forward traversal: 2 rows
              src_decision_id        src_decision_type         edge_type  \
0  dp_allocate_media_budget_1  media_budget_allocation  allocated_budget   
1  dp_allocate_media_budget_1  media_budget_allocation  allocated_budget   

   mako_scoreValue                    dst_adUnitId  dst_adUnitName  \
0             0.82     yahoo_mail_display_standard      Yahoo Mail   
1             1.23  yahoo_homepage_display_premium  Yahoo Homepage   

  dst_adUnitSize dst_adUnitPosition  
0        display           Standard  
1        display  Premium Placement  


In [20]:
lineage_gql = compile_lineage_gql(spec=spec, project_id=PROJECT_ID, dataset_id=SCRATCH_DATASET, relationship_name="sup_YahooAdUnitEvolvedFrom")
print("=== Lineage GQL ===")
print(lineage_gql)

try:
    job = bq.query(lineage_gql, job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("session_id", "STRING", "sess-synthetic-B"),
            bigquery.ScalarQueryParameter("result_limit", "INT64", 50),
        ]
    ))
    lineage_results = job.to_dataframe()
    print(f"\nLineage traversal: {len(lineage_results)} rows")
    if len(lineage_results) > 0:
        print(lineage_results.head(10))
    else:
        print("No lineage rows returned")
except Exception as e:
    print(f"Lineage query failed: {e}")

=== Lineage GQL ===
GRAPH `test-project-0728-467323.agent_analytics_v5_8ec7050a.YMGO_Context_Graph_V3`
MATCH
  (prev:sup_YahooAdUnit)-[eyauef:sup_YahooAdUnitEvolvedFrom]->(cur:sup_YahooAdUnit)
WHERE cur.session_id = @session_id
RETURN
  prev.adUnitId AS prior_adUnitId,
  prev.adUnitName AS prior_adUnitName,
  prev.adUnitSize AS prior_adUnitSize,
  prev.adUnitPosition AS prior_adUnitPosition,
  eyauef.from_session_id,
  eyauef.to_session_id,
  eyauef.event_time,
  eyauef.changed_properties,
  cur.adUnitId AS current_adUnitId,
  cur.adUnitName AS current_adUnitName,
  cur.adUnitSize AS current_adUnitSize,
  cur.adUnitPosition AS current_adUnitPosition
ORDER BY eyauef.event_time DESC
LIMIT @result_limit




Lineage traversal: 1 rows
                   prior_adUnitId prior_adUnitName prior_adUnitSize  \
0  yahoo_homepage_display_premium   Yahoo Homepage          display   

  prior_adUnitPosition    from_session_id     to_session_id  \
0    Premium Placement  adcp-033c95d7a97d  sess-synthetic-B   

                        event_time         changed_properties  \
0 2026-04-14 05:59:10.452196+00:00  adUnitName,adUnitPosition   

                 current_adUnitId           current_adUnitName  \
0  yahoo_homepage_display_premium  Yahoo Homepage (Redesigned)   

  current_adUnitSize current_adUnitPosition  
0            display                    BTF  


In [21]:
# Clean up the scratch dataset
bq.delete_dataset(
    f"{PROJECT_ID}.{SCRATCH_DATASET}",
    delete_contents=True,
    not_found_ok=True,
)
print(f"Deleted scratch dataset: {SCRATCH_DATASET}")

Deleted scratch dataset: agent_analytics_v5_8ec7050a


## Summary

This notebook demonstrated the **Context Graph V5** pipeline end-to-end:

| Step | What it does | Key API |
|------|-------------|---------|
| **TTL Import** | Bootstrap a GraphSpec YAML from an OWL/Turtle file | `ttl_import()`, `ttl_resolve()` |
| **Mixed Extraction** | Combine deterministic structured extractors with AI.GENERATE | `OntologyGraphManager(extractors=...)` |
| **Batch Materialization** | Insert rows via staging tables with full status reporting | `OntologyMaterializer(write_mode="batch_load")`, `materialize_with_status()` |
| **Temporal Lineage** | Detect and materialize cross-session entity evolution | `detect_lineage_edges()` |
| **GQL Queries** | Forward traversal and lineage traversal over the property graph | `compile_showcase_gql()`, `compile_lineage_gql()` |

**What was verified:**
- TTL import produces valid YAML with entities, relationships, and primary keys.
- `ttl_resolve` replaces FILL_IN placeholders with user-supplied keys.
- The BKA structured extractor produces deterministic `ExtractedNode` instances.
- `batch_load` mode materializes rows via staging tables with `TableStatus` reporting.
- `detect_lineage_edges` finds shared-key entities with changed properties across sessions.
- Lineage edges are materialized and queryable via `compile_lineage_gql`.

**Limitations:**
- The lineage demo uses a synthetic Session B; real lineage requires multiple real sessions.
- GQL query results depend on the actual data extracted from your ADK telemetry.
- The scratch dataset auto-expires tables after 1 hour, but explicit cleanup is still recommended.
This notebook demonstrates the V5 feature path end to end; it is a deterministic demo, not a proof that current real-world telemetry always yields lineage overlap or perfectly stable AI extraction.
